In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ML ve NLP kütüphaneleri
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.pipeline import Pipeline
import mlflow
import mlflow.sklearn

# Veriyi oku
df = pd.read_csv('../data/processed/telecom_customer_tickets.csv')
print(f"Toplam Yüklenen Bilet: {len(df):,}")
df.head(3)

Toplam Yüklenen Bilet: 60,377


,tweet_id,author_id,created_at,text,cleaned_text
0,3,115712,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,sent several private messages one responding u...
1,8,115712,Tue Oct 31 21:45:10 +0000 2017,@sprintcare is the worst customer service,worst customer service
2,12,115713,Tue Oct 31 22:04:47 +0000 2017,@sprintcare You gonna magically change your co...,gonna magically change connectivity whole family


In [6]:
# Telekom etiketleme kuralları
def assign_category(text: str) -> str:
    text = str(text).lower()
    
    # Ağ / Sinyal / İnternet
    if any(k in text for k in ['network', 'signal', 'service', 'slow', 'down', 'internet', 'data', '4g', '5g', 'lte', 'wifi', 'connection', 'coverage', 'dropped', 'outage']):
        return 'Network/Signal'
    
    # Fatura / Ödeme
    elif any(k in text for k in ['bill', 'charged', 'charge', 'payment', 'money', 'refund', 'fee', 'price', 'cost', 'plan', 'contract', 'autopay', 'account balance', 'paid']):
        return 'Billing/Payment'
    
    # Donanım / SIM / Cihaz
    elif any(k in text for k in ['phone', 'iphone', 'android', 'device', 'sim', 'screen', 'upgrade', 'order', 'delivery', 'unlocked', 'activation', 'battery', 'esim']):
        return 'Device/SIM'
    
    # Müşteri Hizmetleri / Genel
    else:
        return 'Customer Service'

# Etiketleme fonksiyonunu uygula
df['category'] = df['cleaned_text'].apply(assign_category)

# Kategori dağılımını incele
print("--- Kategori Dağılımı ---")
print(df['category'].value_counts())
print("\nYüzdelik Dağılım:")
print(df['category'].value_counts(normalize=True) * 100)

--- Kategori Dağılımı ---
category
Customer Service    27843
Network/Signal      14897
Device/SIM          10330
Billing/Payment      7307
Name: count, dtype: int64

Yüzdelik Dağılım:
category
Customer Service    46.115243
Network/Signal      24.673303
Device/SIM          17.109164
Billing/Payment     12.102291
Name: proportion, dtype: float64


In [7]:
# X (Özellikler - Metin) ve y (Hedef - Kategori)
X = df['cleaned_text']
y = df['category']

# %80 Eğitim, %20 Test olarak ayır (stratify=y ile sınıf dağılımı korunur)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Eğitim Seti Boyutu: {len(X_train):,}")
print(f"Test Seti Boyutu: {len(X_test):,}")

Eğitim Seti Boyutu: 48,301
Test Seti Boyutu: 12,076
